# 08 - Risk Management Module
Position sizing, daily loss limits, expectancy tracking

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime, date
import warnings
warnings.filterwarnings('ignore')

## 1. Position Sizing
Calculate lot size based on account balance, risk %, and SL distance

In [9]:
class PositionSizer:
    def __init__(self, account_balance=4654.85, risk_percent=1.0):
        self.balance = account_balance
        self.risk_percent = risk_percent
        self.pip_value_usd = 1.0  # 1 pip = $1 for 10k units EURUSD

    def calculate(self, sl_pips, lot_size_step=0.01):
        risk_amount = self.balance * (self.risk_percent / 100)
        raw_lot_size = risk_amount / (sl_pips * 10)  # 10 units per pip per standard lot
        lot_size = round(raw_lot_size / lot_size_step) * lot_size_step
        lot_size = max(lot_size, 0.01)
        return {
            'balance': self.balance,
            'risk_percent': self.risk_percent,
            'risk_amount': round(risk_amount, 2),
            'sl_pips': sl_pips,
            'lot_size': lot_size,
            'risk_per_lot': round(sl_pips * 10, 2),
            'actual_risk': round(lot_size * sl_pips * 10, 2)
        }

sizer = PositionSizer()

for sl in [10, 20, 30, 40, 50]:
    r = sizer.calculate(sl)
    print(f"SL={sl}pip | Risk=${r['risk_amount']} | Lot={r['lot_size']:.2f} | Actual=${r['actual_risk']}")

SL=10pip | Risk=$46.55 | Lot=0.47 | Actual=$47.0
SL=20pip | Risk=$46.55 | Lot=0.23 | Actual=$46.0
SL=30pip | Risk=$46.55 | Lot=0.16 | Actual=$48.0
SL=40pip | Risk=$46.55 | Lot=0.12 | Actual=$48.0
SL=50pip | Risk=$46.55 | Lot=0.09 | Actual=$45.0


In [10]:
# Simulate different account sizes
for bal in [1000, 5000, 10000, 50000]:
    s = PositionSizer(bal, 1.0)
    r = s.calculate(20)
    print(f"Balance=${bal} | Lot={r['lot_size']:.2f} | Risk=${r['risk_amount']} | SL=20pip")

Balance=$1000 | Lot=0.05 | Risk=$10.0 | SL=20pip
Balance=$5000 | Lot=0.25 | Risk=$50.0 | SL=20pip
Balance=$10000 | Lot=0.50 | Risk=$100.0 | SL=20pip
Balance=$50000 | Lot=2.50 | Risk=$500.0 | SL=20pip


## 2. Daily Loss / Risk Tracker
Track daily P&L, enforce max loss limit

In [11]:
class DailyTracker:
    def __init__(self, max_daily_loss_pct=2.0, max_concurrent=2, starting_balance=4654.85):
        self.max_loss_pct = max_daily_loss_pct
        self.max_concurrent = max_concurrent
        self.balance = starting_balance
        self.daily_pnl = 0
        self.current_date = date.today()
        self.open_trades = 0
        self.trades = []

    def new_day(self, new_balance=None):
        old_pnl = self.daily_pnl
        self.daily_pnl = 0
        self.current_date = date.today()
        if new_balance:
            self.balance = new_balance
        return old_pnl

    def can_trade(self):
        max_loss = self.balance * (self.max_loss_pct / 100)
        if self.daily_pnl <= -max_loss:
            return False, f'Daily loss limit hit (${abs(self.daily_pnl):.2f} >= ${max_loss:.2f})'
        if self.open_trades >= self.max_concurrent:
            return False, f'Max concurrent trades ({self.max_concurrent}) reached'
        return True, 'OK'

    def open(self):
        if self.can_trade()[0]:
            self.open_trades += 1

    def close(self, pnl):
        self.open_trades = max(0, self.open_trades - 1)
        self.daily_pnl += pnl
        self.balance += pnl
        self.trades.append({'date': self.current_date, 'pnl': pnl, 'balance': self.balance})

tracker = DailyTracker()

for pnl in [20, -30, 15, -40, -50]:
    ok, msg = tracker.can_trade()
    if ok:
        tracker.open()
        tracker.close(pnl)
    else:
        print(f'BLOCKED: {msg}')
    print(f"  Balance=${tracker.balance:.2f} | Daily={tracker.daily_pnl:+.2f} | Open={tracker.open_trades}")

print(f'\nFinal balance: ${tracker.balance:.2f}')

  Balance=$4674.85 | Daily=+20.00 | Open=0
  Balance=$4644.85 | Daily=-10.00 | Open=0
  Balance=$4659.85 | Daily=+5.00 | Open=0
  Balance=$4619.85 | Daily=-35.00 | Open=0
  Balance=$4569.85 | Daily=-85.00 | Open=0

Final balance: $4569.85


## 3. Expectancy & Profit Factor Calculator

In [12]:
class PerformanceTracker:
    def __init__(self):
        self.trades = []

    def add_trade(self, entry_date, direction, entry, sl, tp, exit_price, exit_reason='SL/TP'):
        pips = (exit_price - entry) * 10000
        if direction == 'sell':
            pips = -pips
        pnl = pips * 1.0  # $1 per pip for 10k units
        self.trades.append({
            'date': entry_date,
            'direction': direction,
            'entry': entry,
            'exit': exit_price,
            'pips': round(pips, 1),
            'pnl': round(pnl, 2),
            'exit_reason': exit_reason
        })

    def summary(self):
        if len(self.trades) == 0:
            return {'trades': 0}
        df = pd.DataFrame(self.trades)
        wins = df[df['pnl'] > 0]
        losses = df[df['pnl'] < 0]
        gross_profit = wins['pnl'].sum() if len(wins) > 0 else 0
        gross_loss = abs(losses['pnl'].sum()) if len(losses) > 0 else 0
        pf = gross_profit / gross_loss if gross_loss > 0 else float('inf')
        win_rate = len(wins) / len(df) * 100 if len(df) > 0 else 0
        avg_win = wins['pnl'].mean() if len(wins) > 0 else 0
        avg_loss = abs(losses['pnl'].mean()) if len(losses) > 0 else 0
        expectancy = (win_rate/100 * avg_win) - ((1-win_rate/100) * avg_loss)
        return {
            'trades': len(df),
            'wins': len(wins),
            'losses': len(losses),
            'win_rate': round(win_rate, 1),
            'gross_profit': round(gross_profit, 2),
            'gross_loss': round(gross_loss, 2),
            'profit_factor': round(pf, 2),
            'avg_win': round(avg_win, 2),
            'avg_loss': round(avg_loss, 2),
            'expectancy': round(expectancy, 2),
            'net_pnl': round(df['pnl'].sum(), 2)
        }

pt = PerformanceTracker()

for i in range(30):
    pt.add_trade('2026-05-01', 'buy', 1.1000, 1.0980, 1.1040, 1.1040)
for i in range(20):
    pt.add_trade('2026-05-01', 'sell', 1.1000, 1.1020, 1.0960, 1.1020)

s = pt.summary()
for k, v in s.items():
    print(f"{k}: {v}")

trades: 50
wins: 30
losses: 20
win_rate: 60.0
gross_profit: 1200.0
gross_loss: 400.0
profit_factor: 3.0
avg_win: 40.0
avg_loss: 20.0
expectancy: 16.0
net_pnl: 800.0


## 4. Full Risk Manager — combines all

In [13]:
class RiskManager:
    def __init__(self, balance=4654.85, risk_per_trade=1.0, max_daily_loss=2.0, max_concurrent=2):
        self.sizer = PositionSizer(balance, risk_per_trade)
        self.tracker = DailyTracker(max_daily_loss, max_concurrent, balance)
        self.performance = PerformanceTracker()

    def get_signal(self, sl_pips):
        ok, msg = self.tracker.can_trade()
        if not ok:
            return {'allowed': False, 'reason': msg}
        sizing = self.sizer.calculate(sl_pips)
        return {'allowed': True, 'sizing': sizing}

    def execute(self, sl_pips):
        signal = self.get_signal(sl_pips)
        if not signal['allowed']:
            return signal
        self.tracker.open()
        return signal

    def close_trade(self, pnl):
        self.tracker.close(pnl)

    def report(self):
        print('=' * 45)
        print('RISK MANAGER REPORT')
        print('=' * 45)
        print(f'Balance:        ${self.tracker.balance:.2f}')
        print(f'Daily P&L:      {self.tracker.daily_pnl:+.2f}')
        print(f'Open trades:    {self.tracker.open_trades}/{self.tracker.max_concurrent}')
        perf = self.performance.summary()
        if perf['trades'] > 0:
            print(f'Total trades:   {perf["trades"]} ({perf["wins"]}W / {perf["losses"]}L)')
            print(f'Win rate:       {perf["win_rate"]}%')
            print(f'Profit Factor:  {perf["profit_factor"]}')
            print(f'Expectancy:     ${perf["expectancy"]}/trade')
            print(f'Net P&L:        ${perf["net_pnl"]:.2f}')
        print('=' * 45)

rm = RiskManager()

for i in range(10):
    signal = rm.execute(30)
    pnl = np.random.choice([40, -30], p=[0.4, 0.6])
    rm.close_trade(pnl)

rm.report()

RISK MANAGER REPORT
Balance:        $4564.85
Daily P&L:      -90.00
Open trades:    0/2


In [14]:
# Exportable RiskManager for Week 5 backtest
print("""
Export risk_management module for backtest:
  from risk_management import RiskManager, PositionSizer

Usage:
  rm = RiskManager(balance=4654.85, risk_per_trade=1.0,
                   max_daily_loss=2.0, max_concurrent=2)
  signal = rm.execute(sl_pips=25)
  if signal['allowed']:
      # enter trade
      # ...
      rm.close_trade(pnl_value)
  print(rm.report())
""")


Export risk_management module for backtest:
  from risk_management import RiskManager, PositionSizer

Usage:
  rm = RiskManager(balance=4654.85, risk_per_trade=1.0,
                   max_daily_loss=2.0, max_concurrent=2)
  signal = rm.execute(sl_pips=25)
  if signal['allowed']:
      # enter trade
      # ...
      rm.close_trade(pnl_value)
  print(rm.report())

